## Load Dataset

In [21]:
import kagglehub
path = kagglehub.dataset_download("uciml/pima-indians-diabetes-database")

Using Colab cache for faster access to the 'pima-indians-diabetes-database' dataset.


In [22]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/pima-indians-diabetes-database/diabetes.csv


In [23]:
import pandas as pd
df = pd.read_csv('/kaggle/input/pima-indians-diabetes-database/diabetes.csv')

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [25]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [26]:
df.sample(5)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
111,8,155,62,26,495,34.0,0.543,46,1
406,4,115,72,0,0,28.9,0.376,46,1
207,5,162,104,0,0,37.7,0.151,52,1
272,3,122,78,0,0,23.0,0.254,40,0
146,9,57,80,37,0,32.8,0.096,41,0


In [27]:
# finding missing values
df.isna().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [28]:
# defining features and labels
X = df.drop(columns=["Outcome"])

y = df["Outcome"]

In [29]:
# checking the weightage
df["Outcome"].value_counts(normalize=True)*100

,proportion
Outcome,
0,65.104167
1,34.895833


In [30]:
# this ratio might create CLASS IMBALANCE.
# we need to stratify it
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
    )

## XGBoost

In [34]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

xgb_model = XGBClassifier(
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,          # Shorter trees help prevent overfitting on smaller datasets
    eval_metric='logloss' # Silences modern deprecation warnings
)

xgb_model.fit(X_train, y_train)

# Predict on the test set
xgb_predictions = xgb_model.predict(X_test)

# Evaluate the performance
xgb_accuracy = accuracy_score(y_test, xgb_predictions)
xgb_precision = precision_score(y_test, xgb_predictions)
xgb_recall = recall_score(y_test, xgb_predictions)
xgb_f1 = f1_score(y_test, xgb_predictions)

print(f"XGBoost Accuracy:  {xgb_accuracy:.4f}")
print(f"XGBoost Precision: {xgb_precision:.4f}")
print(f"XGBoost Recall:    {xgb_recall:.4f}")
print(f"XGBoost F1-Score:  {xgb_f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_predictions))

XGBoost Accuracy:  0.7597
XGBoost Precision: 0.6735
XGBoost Recall:    0.6111
XGBoost F1-Score:  0.6408

Confusion Matrix:
[[84 16]
 [21 33]]


## Gradio UI

In [36]:
import gradio as gr
import numpy as np

def predict_diabetes(pregnancies, glucose, bloodpressure, skinthickness, insulin, bmi, diabetespedigreefunction, age):
    # Create a numpy array from the input values
    input_data = np.array([
        pregnancies, glucose, bloodpressure, skinthickness,
        insulin, bmi, diabetespedigreefunction, age
    ]).reshape(1, -1)

    # Make prediction using the trained random forest model
    prediction = xgb_model.predict(input_data)

    # Map prediction to human-readable label
    return "Diabetic" if prediction[0] == 1 else "Not Diabetic"

# Build the UI using Blocks
with gr.Blocks() as interface:
    gr.Markdown("# Pima Indians Diabetes Prediction")
    gr.Markdown("### XGBoost")
    gr.Markdown("Enter patient details to predict if they have diabetes.")

    with gr.Row():
        with gr.Column():
            pregnancies = gr.Number(label="Pregnancies")
            glucose = gr.Number(label="Glucose")
            bloodpressure = gr.Number(label="BloodPressure")
            skinthickness = gr.Number(label="SkinThickness")
        with gr.Column():
            insulin = gr.Number(label="Insulin")
            bmi = gr.Number(label="BMI")
            diabetespedigreefunction = gr.Number(label="DiabetesPedigreeFunction")
            age = gr.Number(label="Age")

    submit_btn = gr.Button("Predict")
    output = gr.Label(label="Prediction")

    # Connect the button to the prediction function
    submit_btn.click(
        fn=predict_diabetes,
        inputs=[pregnancies, glucose, bloodpressure, skinthickness, insulin, bmi, diabetespedigreefunction, age],
        outputs=output
    )

# Launch the interface
interface.launch(inline=True, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://50e15bffac9b993667.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
